In [17]:
import pandas as pd

data = pd.read_excel(r"C:\Users\vegev\Downloads\AllElectronics_Dataset.xlsx")
print("\nClass Distribution:")
print(data.iloc[:, -1].value_counts())

import math
def entropy(column):
    counts = column.value_counts()
    total = len(column)
    e = 0
    for count in counts:
        p = count / total
        e = e - p * math.log2(p)
    return e
print("Entropy =", entropy(data["Class_Buys_Computer"]))
#IG
def information_gain(data, attribute):
    total_entropy = entropy(data["Class_Buys_Computer"])
    values = data[attribute].unique()
    attribute_entropy = 0
    for value in values:
        subset = data[data[attribute] == value]
        weight = len(subset) / len(data)
        subset_entropy = entropy(subset["Class_Buys_Computer"])
        attribute_entropy += weight * subset_entropy
    gain = total_entropy - attribute_entropy
    return gain
attributes = data.columns[1:-1]   


print("Information Gain of each feature\n")
for attribute in attributes:
    gain = information_gain(data, attribute)
    print(attribute, "=",gain)   
best_attribute = ""
best_gain = -1
for attribute in attributes:
    gain = information_gain(data, attribute)
    if gain > best_gain:
        best_gain = gain
        best_attribute = attribute
print("Best Attribute :", best_attribute)
print("Information Gain :", best_gain)
values = data[best_attribute].unique()
for value in values:
    subset = data[data[best_attribute] == value]
    print(best_attribute, "=", value)
    print(subset)

def build_tree(data):

    if len(data["Class_Buys_Computer"].unique()) == 1:
        return data["Class_Buys_Computer"].iloc[0]

    # If no attributes are left
    if len(data.columns) <= 2:
        return data["Class_Buys_Computer"].mode()[0]
    best_attribute = ""
    best_gain = -1
    attributes = data.columns[1:-1]

    for attribute in attributes:
        gain = information_gain(data, attribute)
        if gain > best_gain:
            best_gain = gain
            best_attribute = attribute
    tree = {best_attribute: {}}
    values = data[best_attribute].unique()
    for value in values:
        subset = data[data[best_attribute] == value]
        subset = subset.drop(columns=[best_attribute])
        tree[best_attribute][value] = build_tree(subset)

    return tree
tree = build_tree(data)    
print(tree)


Class Distribution:
Class_Buys_Computer
yes    9
no     5
Name: count, dtype: int64
Entropy = 0.9402859586706311
Information Gain of each feature

Age = 0.24674981977443933
Income = 0.02922256565895487
Student = 0.15183550136234159
Credit_Rating = 0.04812703040826949
Best Attribute : Age
Information Gain : 0.24674981977443933
Age = youth
    RID    Age  Income Student Credit_Rating Class_Buys_Computer
0     1  youth    high      no          fair                  no
1     2  youth    high      no     excellent                  no
7     8  youth  medium      no          fair                  no
8     9  youth     low     yes          fair                 yes
10   11  youth  medium     yes     excellent                 yes
Age = middle_aged
    RID          Age  Income Student Credit_Rating Class_Buys_Computer
2     3  middle_aged    high      no          fair                 yes
6     7  middle_aged     low     yes     excellent                 yes
11   12  middle_aged  medium      no  

In [22]:
def split_information(data, attribute):
    values = data[attribute].unique()
    split_info = 0
    for value in values:
        subset = data[data[attribute] == value]
        weight = len(subset) / len(data)
        split_info -= weight * math.log2(weight)
    return split_info
def gain_ratio(data, attribute): #gian ratio
    gain = information_gain(data, attribute)
    split_info = split_information(data, attribute)
    if split_info == 0:
        return 0
    return gain / split_info


attributes = data.columns[1:-1] #finding best gain raioi

print("Gain Ratio of all Atribute\n")
for attribute in attributes:
    ratio = gain_ratio(data, attribute)
    print(attribute, "=", round(ratio,4))

best_attribute = ""
best_ratio = -1

for attribute in attributes:
    ratio = gain_ratio(data, attribute)
    if ratio > best_ratio:
        best_ratio = ratio
        best_attribute = attribute

print("Best Attribute :", best_attribute)
print("Gain Ratio :",best_ratio)

def build_c45_tree(data):
    if len(data["Class_Buys_Computer"].unique()) == 1:
        return data["Class_Buys_Computer"].iloc[0]
    if len(data.columns) <= 2:
        return data["Class_Buys_Computer"].mode()[0]

    attributes = data.columns[1:-1]

    best_attribute = ""
    best_ratio = -1
    for attribute in attributes:
        ratio = gain_ratio(data, attribute)
        if ratio > best_ratio:
            best_ratio = ratio
            best_attribute = attribute
    tree = {best_attribute: {}}
    values = data[best_attribute].unique()

    for value in values:
        subset = data[data[best_attribute] == value]
        subset = subset.drop(columns=[best_attribute])
        tree[best_attribute][value] = build_c45_tree(subset)
    return tree

c45_tree = build_c45_tree(data)
print(c45_tree)

Gain Ratio of all Atribute

Age = 0.1564
Income = 0.0188
Student = 0.1518
Credit_Rating = 0.0488
Best Attribute : Age
Gain Ratio : 0.15642756242117528
{'Age': {'youth': {'Student': {'no': 'no', 'yes': 'yes'}}, 'middle_aged': 'yes', 'senior': {'Credit_Rating': {'fair': 'yes', 'excellent': 'no'}}}}


In [24]:

# Gini Index
def gini(column):
    counts = column.value_counts()
    total = len(column)
    gini_index = 1
    for count in counts:
        p = count / total
        gini_index -= p ** 2
    return gini_index
print("Gini Index =", round(gini(data["Class_Buys_Computer"]),4))

def gini_gain(data, attribute):
    total_gini = gini(data["Class_Buys_Computer"])
    values = data[attribute].unique()
    attribute_gini = 0
    for value in values:
        subset = data[data[attribute] == value]
        weight = len(subset) / len(data)
        subset_gini = gini(subset["Class_Buys_Computer"])
        attribute_gini += weight * subset_gini
    gain = total_gini - attribute_gini
    return gain
    
attributes = data.columns[1:-1]
print("Gini Gain of each Attribute\n")
for attribute in attributes:
    gain = gini_gain(data, attribute)
    print(attribute, "=", round(gain,4))

best_attribute = ""
best_gain = -1
for attribute in attributes:
    gain = gini_gain(data, attribute)
    if gain > best_gain:
        best_gain = gain
        best_attribute = attribute
print("\nBest Attribute :", best_attribute)
print("Gini Gain :", round(best_gain,4))

values = data[best_attribute].unique()
for value in values:
    subset = data[data[best_attribute] == value]
    print("\n", best_attribute, "=", value)
    print(subset)

def best_attribute_cart(data):
    attributes = data.columns[1:-1]
    best_attribute = ""
    best_gain = -1
    for attribute in attributes:
        gain = gini_gain(data, attribute)
        if gain > best_gain:
            best_gain = gain
            best_attribute = attribute
    return best_attribute
def build_cart_tree(data):
    if len(data["Class_Buys_Computer"].unique()) == 1:
        return data["Class_Buys_Computer"].iloc[0]
    if len(data.columns) <= 2:
        return data["Class_Buys_Computer"].mode()[0]
    best_attribute = best_attribute_cart(data)
    tree = {best_attribute: {}}
    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value]
        subset = subset.drop(columns=[best_attribute])
        tree[best_attribute][value] = build_cart_tree(subset)
    return tree
print(tree)

Gini Index = 0.4592
Gini Gain of each Attribute

Age = 0.1163
Income = 0.0187
Student = 0.0918
Credit_Rating = 0.0306

Best Attribute : Age
Gini Gain : 0.1163

 Age = youth
    RID    Age  Income Student Credit_Rating Class_Buys_Computer
0     1  youth    high      no          fair                  no
1     2  youth    high      no     excellent                  no
7     8  youth  medium      no          fair                  no
8     9  youth     low     yes          fair                 yes
10   11  youth  medium     yes     excellent                 yes

 Age = middle_aged
    RID          Age  Income Student Credit_Rating Class_Buys_Computer
2     3  middle_aged    high      no          fair                 yes
6     7  middle_aged     low     yes     excellent                 yes
11   12  middle_aged  medium      no     excellent                 yes
12   13  middle_aged    high     yes          fair                 yes

 Age = senior
    RID     Age  Income Student Credit_Rating C

In [27]:
import time

train_size = int(0.8 * len(data))
train_data = data.iloc[:train_size]
test_data = data.iloc[train_size:]

print("Training Records :", len(train_data))
print("Testing Records :", len(test_data))

start = time.time()
#tree = build_tree(train_data)          # ID3
tree = build_c45_tree(train_data)    # c4.5
# tree = build_cart_tree(train_data)   # CART
end = time.time()

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree
    attribute = list(tree.keys())[0]
    value = sample[attribute]
    if value not in tree[attribute]:
        return train_data["Class_Buys_Computer"].mode()[0]
    return predict(tree[attribute][value], sample)

TP = TN = FP = FN = 0
for i in range(len(test_data)):
    sample = test_data.iloc[i]
    actual = sample["Class_Buys_Computer"]
    predicted = predict(tree, sample)
    if actual == "yes" and predicted == "yes":
        TP += 1
    elif actual == "no" and predicted == "no":
        TN += 1
    elif actual == "no" and predicted == "yes":
        FP += 1
    else:
        FN += 1


accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1_score = (2 * precision * recall) / (precision + recall)
print("\nConfusion Matrix")
print("TP =", TP)
print("TN =", TN)
print("FP =", FP)
print("FN =", FN)
print("\nAccuracy =", round(accuracy,4))
print("Precision =", round(precision,4))
print("Recall =", round(recall,4))
print("F1 Score =", round(f1_score,4))
print("Training Time =", round(end-start,6), "seconds")

Training Records : 11
Testing Records : 3

Confusion Matrix
TP = 2
TN = 0
FP = 1
FN = 0

Accuracy = 0.6667
Precision = 0.6667
Recall = 1.0
F1 Score = 0.8
Training Time = 0.069041 seconds
